# 🍽️ トレー食品残渣 分類モデル

EfficientNet-B0 の転移学習による **clean（残渣なし） / dirty（残渣あり）** 2クラス分類です。

---
### 📋 使い方（3ステップ）
1. **ランタイム → ランタイムのタイプを変更 → T4 GPU** に設定
2. `data.zip` を用意して Step 2 でアップロード
3. あとはセルを上から順に実行するだけ

> `data.zip` は `prepare_data.py` を実行すると自動で作成されます

## ⚙️ Step 1: 環境セットアップ

In [ ]:
import torch, torchvision
print(f'PyTorch     : {torch.__version__}')
print(f'torchvision : {torchvision.__version__}')
print(f'GPU 使用可能 : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')

In [ ]:
import os, copy, random, zipfile, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import EfficientNet_B0_Weights
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用デバイス: {DEVICE}')

## 📤 Step 2: data.zip をアップロード

ローカルで `prepare_data.py` を実行して作成した **`data.zip`** をアップロードしてください。

ZIP の中身はこの構造になっています:
```
data/
├── train/
│   ├── clean/   clean_001.png ...
│   └── dirty/   dirty_001.png ...
└── val/
    ├── clean/
    └── dirty/
```

In [ ]:
from google.colab import files
import shutil

print('data.zip を選択してください')
uploaded = files.upload()

# 既存の data/ を削除してからクリーンに展開
shutil.rmtree('data', ignore_errors=True)

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')
print('展開完了\n')

total = 0
for split in ['train', 'val']:
    for label in ['clean', 'dirty']:
        p = Path(f'data/{split}/{label}')
        n = len(list(p.glob('*'))) if p.exists() else 0
        total += n
        print(f'  data/{split}/{label}: {n} 枚')
print(f'\n合計: {total} 枚')

In [ ]:
# アップロードした画像を確認
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, label in enumerate(['clean', 'dirty']):
    img_paths = sorted(Path(f'data/train/{label}').glob('*'))[:4]
    for col, p in enumerate(img_paths):
        axes[row][col].imshow(Image.open(p))
        axes[row][col].set_title(
            f'{label}\n{p.name}',
            color='steelblue' if label=='clean' else 'tomato',
            fontsize=9
        )
        axes[row][col].axis('off')
plt.suptitle('学習データのサンプル（各クラス最大4枚）', fontsize=13)
plt.tight_layout()
plt.show()

## 🔧 Step 3: 学習設定

> **少量データ（各クラス10枚以下）向けの推奨設定になっています。**  
> データが増えたら `EPOCHS` を大きくしてください。

In [ ]:
# ========== 変更したい場合はここを編集 ==========
IMG_SIZE    = 224   # 入力サイズ（変更不要）
BATCH_SIZE  = 8     # 少量データなので小さめに設定
EPOCHS      = 40    # 少量データなので多めに回す
LR          = 5e-4  # 少量データは低めの学習率が安定しやすい
UNFREEZE_AT = 20    # このエポックから全層 fine-tune
# ================================================

print(f'バッチサイズ   : {BATCH_SIZE}')
print(f'エポック数     : {EPOCHS}')
print(f'学習率         : {LR}')
print(f'全層 fine-tune : {UNFREEZE_AT} epoch から')

## 🏋️ Step 4: 学習

In [ ]:
# 少量データ向けに強めのデータ拡張を設定
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 48, IMG_SIZE + 48)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.15),
    transforms.RandomRotation(20),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

image_datasets = {
    'train': datasets.ImageFolder('data/train', train_tf),
    'val'  : datasets.ImageFolder('data/val',   val_tf),
}
dataloaders = {
    k: DataLoader(v, batch_size=BATCH_SIZE, shuffle=(k=='train'), num_workers=2)
    for k, v in image_datasets.items()
}
CLASS_NAMES = image_datasets['train'].classes
print(f'クラス: {CLASS_NAMES}')
print(f'train: {len(image_datasets["train"])} 枚 / val: {len(image_datasets["val"])} 枚')

In [ ]:
def build_model(freeze_base=True):
    model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    if freeze_base:
        for p in model.parameters():
            p.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(in_features, 2),
    )
    return model

model = build_model(freeze_base=True).to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'学習パラメータ: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

In [ ]:
os.makedirs('checkpoints', exist_ok=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=UNFREEZE_AT)

best_acc = 0.0
best_wts = copy.deepcopy(model.state_dict())
history  = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
phase1_epochs = min(UNFREEZE_AT, EPOCHS)

for epoch in range(1, EPOCHS + 1):
    if epoch == phase1_epochs + 1:
        print('\n=== Phase 2: 全層 fine-tune 開始 ===')
        for p in model.parameters():
            p.requires_grad = True
        optimizer = optim.Adam(model.parameters(), lr=LR * 0.1)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS - phase1_epochs)

    print(f'Epoch {epoch:>3}/{EPOCHS}', end='  ')
    for phase in ['train', 'val']:
        model.train() if phase == 'train' else model.eval()
        running_loss, running_corrects = 0.0, 0
        for inputs, labels in dataloaders[phase]:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                if phase == 'train':
                    loss.backward()
                    optimizer.step()
            running_loss     += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels).item()

        n          = len(dataloaders[phase].dataset)
        epoch_loss = running_loss / n
        epoch_acc  = running_corrects / n
        history[f'{phase}_loss'].append(epoch_loss)
        history[f'{phase}_acc'].append(epoch_acc)
        print(f'{phase} loss={epoch_loss:.4f} acc={epoch_acc:.4f}', end='  ')

        if phase == 'val' and epoch_acc > best_acc:
            best_acc = epoch_acc
            best_wts = copy.deepcopy(model.state_dict())
            torch.save(best_wts, 'checkpoints/best_model.pth')
            print('✅ saved', end='')
    print()
    scheduler.step()

model.load_state_dict(best_wts)
print(f'\n🎉 学習完了  Best val acc: {best_acc:.4f}')

## 📊 Step 5: 学習曲線の確認

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_range, history['train_loss'], label='train')
axes[0].plot(epochs_range, history['val_loss'],   label='val')
axes[0].axvline(x=phase1_epochs, color='gray', linestyle='--', label='fine-tune 開始')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(epochs_range, history['train_acc'], label='train')
axes[1].plot(epochs_range, history['val_acc'],   label='val')
axes[1].axvline(x=phase1_epochs, color='gray', linestyle='--', label='fine-tune 開始')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout()
plt.savefig('checkpoints/training_curve.png', dpi=150)
plt.show()

## 📈 Step 6: 評価（混同行列・ROC曲線）

In [ ]:
model.eval()
all_labels, all_preds, all_probs = [], [], []
with torch.no_grad():
    for inputs, labels in dataloaders['val']:
        outputs = model(inputs.to(DEVICE))
        all_probs.extend(F.softmax(outputs, dim=1)[:, 1].cpu().numpy())
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)
all_probs  = np.array(all_probs)

print('=== Classification Report ===')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

cm  = confusion_matrix(all_labels, all_preds)
try:
    auc = roc_auc_score(all_labels, all_probs)
    auc_str = f'{auc:.4f}'
except Exception:
    auc_str = '計算不可（val サンプル不足）'
print(f'AUC-ROC: {auc_str}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(CLASS_NAMES); axes[0].set_yticklabels(CLASS_NAMES)
axes[0].set_xlabel('予測'); axes[0].set_ylabel('正解'); axes[0].set_title('混同行列')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i,j], ha='center', va='center',
                     color='white' if cm[i,j]>cm.max()/2 else 'black', fontsize=18)

try:
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    axes[1].plot(fpr, tpr, label=f'AUC = {auc:.3f}')
    axes[1].plot([0,1],[0,1],'k--')
    axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC Curve'); axes[1].legend()
except Exception:
    axes[1].text(0.5, 0.5, 'val データが少なすぎて\nROC を描画できません',
                 ha='center', va='center', fontsize=12)
    axes[1].axis('off')

plt.tight_layout()
plt.show()

## 🔍 Step 7: 画像をアップロードして判定する

In [ ]:
from google.colab import files as colab_files

INFER_TF = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def predict(img_path):
    img    = Image.open(img_path).convert('RGB')
    tensor = INFER_TF(img).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1)[0].cpu().numpy()
    return img, CLASS_NAMES[probs.argmax()], probs

print('判定したい画像をアップロードしてください（複数可）')
uploaded = colab_files.upload()

n = len(uploaded)
fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1:
    axes = [axes]

for ax, fname in zip(axes, uploaded.keys()):
    img, label, probs = predict(fname)
    ci = CLASS_NAMES.index('clean') if 'clean' in CLASS_NAMES else 0
    di = CLASS_NAMES.index('dirty') if 'dirty' in CLASS_NAMES else 1
    color = 'steelblue' if label == 'clean' else 'tomato'
    title = ('✅ clean（残渣なし）' if label=='clean' else '⚠️  dirty（残渣あり）') \
            + f'\nclean: {probs[ci]:.1%}  dirty: {probs[di]:.1%}'
    ax.imshow(img)
    ax.set_title(title, color=color, fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 💾 Step 8: モデルを保存する

In [ ]:
# Google Drive に保存
from google.colab import drive
drive.mount('/content/drive')
save_path = '/content/drive/MyDrive/tray_classifier_best_model.pth'
shutil.copy('checkpoints/best_model.pth', save_path)
print(f'Google Drive に保存: {save_path}')

In [ ]:
# ローカルへダウンロード
from google.colab import files as colab_files
colab_files.download('checkpoints/best_model.pth')